# Figure S2: PM2.5 comparison-pool sensitivity

Compare the primary PM2.5 results with the sensitivity using all coverage-eligible comparison homes. Both sets use untransformed outcomes; the separate log sensitivity is reported in Table S3.

**Step:** 17. Article figure. See README.md for its data prerequisites.

**Article scope:** the main article and Supplementary Material.

**Inputs:** Read event_results.csv and events.csv. Contrast primary and SI_original_comparisons for the same event and model.

**Outputs:** The named figure in the separate reproduction figures directory, using the original export formats.

**How to use this notebook:** select the Python environment described in `README.md`, restart the kernel,
and run the cells from top to bottom. Edit the configuration cell before running.
The checks below stop execution when the current data no longer match the manuscript specification.

## Locate the code folder and load the shared functions

The repository is found by searching this notebook's folder and its parent folders, so Jupyter can be started anywhere inside the repository. Set `INDOOR_AQ_CODE` only when it starts elsewhere.
It loads only the cells tagged `library` in `00_shared_functions.ipynb` (its demonstration cell is not executed).
Each notebook has its own imports and configuration, so it never depends on another notebook's kernel state.

In [ ]:
# Setup: find the code folder and load the shared functions.
from pathlib import Path
import json
import os
import sys
import types


def locate_code_directory():
    """Locate this repository (the folder with 00_shared_functions.ipynb) from the working directory or INDOOR_AQ_CODE."""
    configured = os.environ.get("INDOOR_AQ_CODE")
    if configured:
        candidates = [Path(configured).expanduser().resolve()]
    else:
        candidates = [Path.cwd(), *Path.cwd().parents]
    for candidate in candidates:
        if ((candidate / "00_shared_functions.ipynb").is_file()
                and (candidate / "5_did_cits").is_dir()):
            return candidate.resolve()
    raise FileNotFoundError("Start Jupyter in the repository folder, or set INDOOR_AQ_CODE to that folder.")

CODE = locate_code_directory()

# Load the cells tagged "library" from 00_shared_functions.ipynb as an in-memory
# module named `shared`, so that `from shared import ...` works below.
# No helper .py file is generated.
_library_path = CODE / "00_shared_functions.ipynb"
_library = types.ModuleType("shared")
_library.__file__ = str(_library_path)
_library.__dict__["CODE"] = CODE
for _cell in json.loads(_library_path.read_text(encoding="utf-8"))["cells"]:
    if _cell["cell_type"] == "code" and "library" in _cell.get("metadata", {}).get("tags", []):
        exec(compile("".join(_cell["source"]), str(_library_path), "exec"), _library.__dict__)
sys.modules["shared"] = _library

from shared import DATA, output_root
print(f"Manuscript code: {CODE}")

## Configure figure paths and resolution

Only presentation and output settings belong here. Keep the manuscript data and event specification fixed when reproducing the final figure.

In [ ]:
# Paths are shared by Python and R. Defaults: data/ for supplied inputs,
# results/ for this run. Set INDOOR_AQ_DATA and INDOOR_AQ_OUTPUT before
# starting Jupyter and R; relative paths are resolved against the repository.
OUTPUT = os.environ.get("INDOOR_AQ_OUTPUT") or None

# Set a lower DPI only for an explicit draft; manuscript exports use 900.
FIGURE_DPI = int(os.environ.get("INDOOR_AQ_FIGURE_DPI", "900"))
os.environ["INDOOR_AQ_OUTPUT"] = str(output_root(OUTPUT))
os.environ["INDOOR_AQ_FIGURE_DPI"] = str(FIGURE_DPI)
# All event-result figures use this run's results folder.


## Load plotting libraries and define the figure settings

The following constants preserve the current figure's outcome order, labels, colors, panel geometry, and reference lines. Numerical methodology and interpretation are documented in the preceding and following cells.

In [ ]:
import os
import sys
from pathlib import Path
from shared import CODE, DATA, output_root, load_hourly as load_manuscript_hourly

import warnings

import matplotlib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from matplotlib import font_manager
from matplotlib.lines import Line2D

# ---- Input data ------------------------------------------------------------
ANALYSIS_DIR = output_root()
RESULTS_PATH = ANALYSIS_DIR / 'tables' / 'event_results.csv'
EVENTS_PATH = ANALYSIS_DIR / 'data' / 'events.csv'

# ---- Output ----------------------------------------------------------------
OUTPUT_DIR = output_root() / 'figures'
OUTPUT_STEM = 'FigS2_PM25_comparison_pool'
EXPORT_FORMATS = ('png', 'pdf')
DPI = int(os.environ.get('INDOOR_AQ_FIGURE_DPI', '900'))

# ---- Font and library version used for the manuscript figure ----------------
FONT_FAMILY = 'Liberation Sans'
FALLBACK_FONT = 'Arial'  # same character widths as Liberation Sans, so the layout is unchanged
# Folders searched for LiberationSans-*.ttf if the font is not installed.
FONT_SEARCH_DIRS = [
    Path(os.environ.get('WINDIR', r'C:\Windows')) / 'Fonts',
    Path(os.environ.get('LOCALAPPDATA', '')) / 'Microsoft' / 'Windows' / 'Fonts',
    Path(r'C:\Program Files\LibreOffice\share\fonts\truetype'),
    # Optional: a folder with LiberationSans-*.ttf (e.g. unzipped from the
    # Liberation fonts release) can be supplied through INDOOR_AQ_FONT_DIR.
    *([Path(os.environ['INDOOR_AQ_FONT_DIR'])] if os.environ.get('INDOOR_AQ_FONT_DIR') else []),
]
EXPECTED_MATPLOTLIB = '3.10'  # other releases may shift text by a sub-pixel

# ---- Rows: retrofit events, top to bottom -----------------------------------
# Display labels use 'dMEV' (UK government usage; 'DMEV' before 12 Oct 2026).
# Event codes such as 'H05_DMEV' are identifiers in the saved results and are unchanged.
EVENTS = ['H05_Loft', 'H05_DMEV', 'H10_Loft', 'H10_DMEV', 'H14_CWI', 'H14_Loft', 'H16_Bundle']
EVENT_LABELS = {
    'H05_Loft': ('Home 5', 'loft insulation'),
    'H05_DMEV': ('Home 5', 'dMEV'),
    'H10_Loft': ('Home 10', 'loft insulation'),
    'H10_DMEV': ('Home 10', 'dMEV'),
    'H14_CWI': ('Home 14', 'CWI'),
    'H14_Loft': ('Home 14', 'loft insulation'),
    'H16_Bundle': ('Home 16', 'combined works'),
}
# Rows are at y = 6 (top) ... 0 (bottom). Light lines separate the four homes.
HOME_SEPARATORS = [4.5, 2.5, 0.5]
Y_LIMITS = (-0.6, 6.6)

# ---- Panels: models ----------------------------------------------------------------
MODELS = ['DiD', 'CITS']

# ---- Comparison pools ------------------------------------------------------------
# (analysis name in the data, line/edge color, marker face color, vertical offset)
POOL_STYLES = [
    ('primary', '#222222', '#222222', 0.16),                   # filled: baseline mean <= 100 µg/m3
    ('SI_original_comparisons', '#6E6E6E', 'white', -0.16),    # open: all coverage-eligible homes
]
# Legend entries as in the published figure: (label, line color, marker face color).
# The primary swatch is #333333, slightly lighter than the plotted #222222.
LEGEND_ENTRIES = [
    ('Primary comparison homes (baseline mean ≤100 µg/m$^3$)', '#333333', '#333333'),
    ('All coverage-eligible comparison homes', '#6E6E6E', 'white'),
]

# ---- Reference lines ------------------------------------------------------------
SEPARATOR_COLOR = '#E3E3E3'
ZERO_LINE_COLOR = '#8C8C8C'
ZERO_LINE_DASH = (0, (3, 2))

# ---- Figure layout (inches and figure fractions) -----------------------------
FIGURE_SIZE = (6.3, 3.9)
SUBPLOT_LAYOUT = dict(left=0.2, right=0.98, top=0.93, bottom=0.25, wspace=0.1)
LEGEND_ANCHOR = (0.58, 0.0)

## Check matplotlib version

Report whether Matplotlib matches the release used for the manuscript. Fonts and text placement may differ across releases even when the underlying data and geometry agree.

In [ ]:
def check_matplotlib_version(expected=EXPECTED_MATPLOTLIB):
    """Warn if matplotlib differs from the release used for the manuscript figure.

    The plotted data are unaffected, but text can be placed a sub-pixel
    differently between releases, so the PNG would not be pixel-identical.
    """
    if not matplotlib.__version__.startswith(expected + '.'):
        warnings.warn(
            f'The manuscript figure was rendered with matplotlib {expected}.x; '
            f'this kernel has {matplotlib.__version__}.')

## Resolve font

Prefer the manuscript font, register an available local font file if needed, and use the stated fallback when unavailable. This affects rendering, not data selection or effect estimation.

In [ ]:
def resolve_font(family=FONT_FAMILY, search_dirs=FONT_SEARCH_DIRS, fallback=FALLBACK_FONT):
    """Return a font family that matplotlib can use, preferring ``family``.

    If ``family`` is not installed, any matching TrueType files found in
    ``search_dirs`` (e.g. LiberationSans-Regular.ttf) are registered for this
    session. If none are found, ``fallback`` is returned with a warning.
    """
    def available():
        return {font.name for font in font_manager.fontManager.ttflist}

    if family in available():
        return family

    pattern = family.replace(' ', '') + '-*.ttf'
    for directory in search_dirs:
        for path in sorted(Path(directory).glob(pattern)):
            font_manager.fontManager.addfont(str(path))

    if family in available():
        return family
    warnings.warn(f'{family} not found; using {fallback}. Glyph shapes will differ slightly.')
    return fallback

## Apply style

Reset Matplotlib defaults before applying the figure's font, line, and export settings. This prevents unrelated notebook styling from carrying over into the manuscript figure.

In [ ]:
def apply_style(font_family):
    """Reset matplotlib to its defaults, then apply the Fig. S2 style.

    Resetting first keeps the output independent of earlier notebook cells
    and of any personal matplotlibrc file.
    """
    plt.rcdefaults()
    plt.rcParams.update({
        'font.family': font_family,
        'font.size': 8,
        'axes.linewidth': 0.6,
        'xtick.major.width': 0.6,
        'ytick.major.width': 0.6,
        'xtick.major.size': 2.5,
        'ytick.major.size': 2.5,
        'axes.spines.top': False,
        'axes.spines.right': False,
        'mathtext.default': 'regular',  # sub/superscripts in the text font, not italic math
        'savefig.facecolor': 'white',
        'pdf.fonttype': 42,             # embed TrueType fonts so PDF text stays editable
    })

## Load pm25 results

Select the original-scale PM2.5 primary and expanded-comparison results. Do not combine log1p coefficients with concentration differences on this axis.

In [ ]:
def load_pm25_results(path=RESULTS_PATH):
    """Load the PM2.5 estimates for both comparison pools.

    Columns used: ``event``, ``model`` ('DiD' or 'CITS'), ``analysis``
    ('primary' or 'SI_original_comparisons'), ``estimate``, ``lower`` and
    ``upper`` (approximate 95% model-based interval).
    """
    results = pd.read_csv(path)
    return results[results['outcome'].eq('pm25')]

## Load events

Read work-start and work-end dates and retain the fixed event order. Construction shading and date labels use these same dates.

In [ ]:
def load_events(path=EVENTS_PATH):
    """Load work start and end dates, indexed by event code."""
    return pd.read_csv(path, parse_dates=['start', 'end']).set_index('event')

## Select estimates

Select one result per event for the requested model or comparison-pool analysis, then reindex to the fixed manuscript event order.

In [ ]:
def select_estimates(results, model, analysis):
    """Rows for one model and comparison pool, ordered as ``EVENTS``."""
    rows = results[results['model'].eq(model) & results['analysis'].eq(analysis)]
    return rows.set_index('event').loc[EVENTS]

## Event tick label

Combine the treated-home and intervention identifiers into the manuscript row label.

In [ ]:
def event_tick_label(event, events):
    """Two-line row label, e.g. 'Home 5, loft insulation' / 'May 2024'."""
    home, measure = EVENT_LABELS[event]
    start = events.loc[event, 'start']
    return f'{home}, {measure}\n{start.month_name()[:3]} {start.year}'

## Draw guides

Add the zero-effect reference and row guides so the direction and uncertainty of each estimate can be read consistently.

In [ ]:
def draw_guides(ax):
    """Draw the separators between homes and the dashed zero line."""
    for y in HOME_SEPARATORS:
        ax.axhline(y, color=SEPARATOR_COLOR, lw=0.6, zorder=0)
    ax.axvline(0, color=ZERO_LINE_COLOR, lw=0.7, ls=ZERO_LINE_DASH, zorder=1)

## Draw estimates

Plot the saved point estimate and both interval limits with the designated marker and horizontal uncertainty line. An interval crossing zero remains visible without being reclassified.

In [ ]:
def draw_estimates(ax, rows, y, color, face_color):
    """Plot circles with asymmetric interval bars; ``face_color='white'`` gives open circles."""
    xerr = [rows['estimate'] - rows['lower'], rows['upper'] - rows['estimate']]
    ax.errorbar(rows['estimate'], y, xerr=xerr, fmt='o', color=color, mfc=face_color, mew=0.9,
                ms=3.8, elinewidth=1.0, capsize=1.8, capthick=0.8, zorder=3)

## Format panel

Apply the panel's labels, tick formatting, limits, and panel letter. Axis display choices do not alter the model or measurement values.

In [ ]:
def format_panel(ax, model, col):
    """Panel title, x label and y range; only the first panel keeps its y axis line."""
    ax.set_title(f'({chr(ord("a") + col)}) {model}', fontsize=8.5, loc='left', pad=5)
    ax.set_xlabel('Estimated PM$_{2.5}$ change (µg/m$^3$)', fontsize=7.5)
    ax.tick_params(labelsize=7)
    ax.set_ylim(*Y_LIMITS)
    ax.spines['left'].set_visible(col == 0)
    if col > 0:
        ax.tick_params(axis='y', length=0)

## Label events

Align event labels with the estimate rows using the shared event order and saved intervention dates.

In [ ]:
def label_events(ax, events, y):
    """Write the event labels on the shared y axis (left-most panel)."""
    ax.set_yticks(y)
    ax.set_yticklabels([event_tick_label(event, events) for event in EVENTS],
                       fontsize=7, linespacing=1.15)
    ax.tick_params(axis='y', length=0)

## Add legend

Explain the series or model markers once in a shared legend. The labels match the manuscript's displayed groups.

In [ ]:
def add_legend(fig):
    """Add the comparison-pool legend (one entry per line) below the panels."""
    handles = [Line2D([], [], color=color, marker='o', mfc=face_color, ms=4, lw=1, label=label)
               for label, color, face_color in LEGEND_ENTRIES]
    fig.legend(handles=handles, loc='lower center', ncol=1, frameon=False,
               fontsize=7.3, bbox_to_anchor=LEGEND_ANCHOR)

## Build figure

Assemble the complete manuscript panel layout from the validated inputs and drawing helpers. This function contains presentation logic and does not fit a statistical model.

In [ ]:
def build_figure(results, events):
    """Build the two forest-plot panels of Fig. S2 and return the figure.

    Parameters
    ----------
    results : pandas.DataFrame
        Output of :func:`load_pm25_results`.
    events : pandas.DataFrame
        Output of :func:`load_events`.
    """
    fig, axes = plt.subplots(1, len(MODELS), figsize=FIGURE_SIZE, sharey=True)
    y = np.arange(len(EVENTS))[::-1]  # first event at the top

    for col, model in enumerate(MODELS):
        ax = axes[col]
        draw_guides(ax)
        for analysis, color, face_color, offset in POOL_STYLES:
            rows = select_estimates(results, model, analysis)
            draw_estimates(ax, rows, y + offset, color, face_color)
        format_panel(ax, model, col)

    label_events(axes[0], events, y)
    add_legend(fig)
    fig.subplots_adjust(**SUBPLOT_LAYOUT)
    return fig

## Save figure

Export the figure to the original file formats. PNG uses the configured resolution (900 dpi by default); PDF and SVG, when included, preserve vector elements. The output directory is separate from the manuscript source figures.

In [ ]:
def save_figure(fig, output_dir=OUTPUT_DIR, stem=OUTPUT_STEM,
                formats=EXPORT_FORMATS, dpi=DPI):
    """Save ``fig`` once per format and return the written paths."""
    output_dir.mkdir(parents=True, exist_ok=True)
    paths = []
    for ext in formats:
        path = output_dir / f'{stem}.{ext}'
        fig.savefig(path, dpi=dpi)
        paths.append(path)
    return paths

## Build and export the figure

Read event_results.csv and events.csv. Contrast primary and SI_original_comparisons for the same event and model. The cells above expose each processing and plotting step for inspection. The following execution writes the figure and prints its actual paths.

In [ ]:
check_matplotlib_version()
apply_style(resolve_font())

results = load_pm25_results()
events = load_events()

fig = build_figure(results, events)
for path in save_figure(fig):
    print(f'Saved {path}')

## Display a compact notebook preview

The preview uses a smaller PNG rendered from the same figure object. The saved publication PNG remains at the configured 900 dpi. Keep methodological detail in the caption when preparing the manuscript.

In [ ]:
import io
from IPython.display import Image, display
_preview = io.BytesIO()
fig.savefig(_preview, format="png", dpi=110, bbox_inches="tight")
display(Image(data=_preview.getvalue()))
plt.close(fig)